In [1]:
print("Antas")

Antas


In [3]:
import pandas as pd
import numpy as np
try:
    import ctgan
    print("CTGAN is installed.")
except ImportError:
    print("CTGAN is not installed.")

try:
    import imblearn
    print("imblearn is installed.")
except ImportError:
    print("imblearn is not installed.")

CTGAN is not installed.
imblearn is installed.


In [4]:
!pip install ctgan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.5 MB/s eta 0:00:00


In [5]:
import pandas as pd
import numpy as np
from ctgan import CTGAN
import warnings
warnings.filterwarnings('ignore')

In [6]:
#LOAD THE DATA
df = pd.read_csv('/content/dataset.csv')
print(df.head())
print("Loading done")

    Age  Gestational Age  Number of sons   Number of daughters  \
0  25.0               26                0                    0   
1  28.0               35                0                    1   
2  26.0               28                0                    0   
3  25.0               28                0                    1   
4  24.0               39                0                    0   

   Total Number of Children       Gravida Female Education Husband Education  \
0                         0  Primigravida       Graduation        Graduation   
1                         1  Multigravida     Intermediate            Matric   
2                         0  Primigravida       Graduation        Graduation   
3                         1  Multigravida       Graduation        Graduation   
4                         0  Primigravida       Graduation        Graduation   

  Working Status Physical Health   ... Feeling down, depressed, or hopeless  \
0      Housewife          Healthy  ...     

In [7]:
# Clean hidden spaces from column names and text
df.columns = df.columns.str.strip()
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()

In [11]:
df.isnull().sum()

,0
Age,2
Gestational Age,0
Number of sons,0
Number of daughters,0
Total Number of Children,0
Gravida,0
Female Education,0
Husband Education,0
Working Status,0
Physical Health,0


In [12]:
df.dropna(inplace=True)

In [13]:
df.isnull().sum()

,0
Age,0
Gestational Age,0
Number of sons,0
Number of daughters,0
Total Number of Children,0
Gravida,0
Female Education,0
Husband Education,0
Working Status,0
Physical Health,0


In [8]:
# Standardize the Target Column
df['Labelling'] = df['Labelling'].map({'Depressed': 1, 'Not': 0, 'nan': np.nan})
df = df.dropna(subset=['Labelling'])
df['Labelling'] = df['Labelling'].astype(int)

# Fix known typos
if 'Current Appereance Acceptance' in df.columns:
    df['Current Appereance Acceptance'] = df['Current Appereance Acceptance'].replace({'YY': 'Yes'})

In [9]:
#GAN IMplemwntation

# CTGAN needs to know which columns are categories (text/discrete) vs continuous numbers (like Age)
discrete_columns = [
    'Gravida', 'Female Education', 'Husband Education', 'Working Status',
    'Physical Health', 'Previous Miscarriage', 'Sufficient Money for Basic Needs',
    'Current Appereance Acceptance', 'Family System', 'Male Gender Preference',
    'Relationship with Mother in-law', 'Labelling'
]

In [14]:
# 3. TRAIN THE GAN (PHASE 2)
print("Initializing CTGAN...")
#epoch =300
ctgan_model = CTGAN(epochs=300, verbose=True)

print("Training the GAN on 14,000 real patients... (This might take a few minutes)")
ctgan_model.fit(df, discrete_columns)

Initializing CTGAN...
Training the GAN on 14,000 real patients... (This might take a few minutes)


Gen. (-01.28) | Discrim. (-00.11): 100%|██████████| 300/300 [17:00<00:00,  3.40s/it]


In [16]:
rows_needed = 16000
print(f"Training complete! Generating {rows_needed} synthetic patients...")
synthetic_data = ctgan_model.sample(rows_needed)

# 5. MERGE AND SAVE
final_30k_df = pd.concat([df, synthetic_data], ignore_index=True)

print(final_30k_df.shape)

Training complete! Generating 16000 synthetic patients...
(29996, 27)


In [18]:
# Save to CSV
output_filename = '30k_ppd_CTGAN.csv'
final_30k_df.to_csv(output_filename, index=False)

print(f"Success! Final dataset shape: {final_30k_df.shape}")
print(output_filename)

Success! Final dataset shape: (29996, 27)
30k_ppd_CTGAN.csv
